In [6]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


In [10]:
!pip install -q \
opentelemetry-api==1.25.0 \
opentelemetry-sdk==1.25.0 \
opentelemetry-exporter-otlp==1.25.0

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.9/59.9 kB 882.6 kB/s eta 0:00:000:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 107.0/107.0 kB 1.9 MB/s eta 0:00:00 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 130.5/130.5 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.5/52.5 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 295.2/295.2 kB 5.4 MB/s eta 0:00:00ta 0:00:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.39.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-adk 1.29.0 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
opentelemetry-instrumentation-asgi 0.55b1 requires opentelemetry-semantic-conventions==0.55b1, but you have opentelemetry-semantic-conventions 0.46b0 which is incompatible.
opentelemetry-instrumentation 0.55b1 require

In [12]:
!pip install -q faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 46.5 MB/s eta 0:00:0000:0100:01


In [7]:
import re
import string

from glob import glob
from gensim.models import Word2Vec
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

import torch
import faiss

from datasets import load_dataset
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

In [50]:
import os
import wandb

wandb_mode = "online" if os.environ.get("WANDB_API_KEY") else "disabled"

run = wandb.init(
    project="23f2000775-t22026",
    name="milestone3-rag",
    mode=wandb_mode,
    config={
        "embedding_model": embedding_model_name,
        "llm": llm_name,
        "top_k": top_k,
        "sample_size": sample_size
    },
    reinit=True
)

wandb.log({
    "MAP@3": rag_map3,
    "Accuracy": rag_accuracy
})

run.finish()

In [22]:
train_path = "/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv"

train = pd.read_csv(train_path)

print(train.shape)
train.head()

(2000, 8)


,id,prompt,A,B,C,D,E,answer
0,1,Pick the best possible answer: What is Martin ...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
1,2,What is accelerator-based light-ion fusion?,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,A
2,3,Determine the correct option: What is the term...,Blueshifting,Redshifting,Reddening,Whitening,Yellowing,C
3,4,Select the most accurate option: What is Marti...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
4,5,Identify the correct statement: What is the co...,"Simultaneity is relative, meaning that two eve...","Simultaneity is relative, meaning that two eve...","Simultaneity is absolute, meaning that two eve...",Simultaneity is a concept that applies only to...,Simultaneity is a concept that applies only to...,A


In [6]:
train.isnull().sum()

id        0
prompt    0
A         0
B         0
C         0
D         0
E         0
answer    0
dtype: int64

In [31]:
text_columns = ['prompt', 'A', 'B', 'C', 'D', 'E']

train[text_columns] = train[text_columns].fillna('')

In [32]:
def clean_text(text):
    text = text.lower()

    for mark in string.punctuation:
        text = text.replace(mark, ' ')

    text = re.sub(r'\s+', ' ', text).strip()

    return text

In [33]:
for column in text_columns:
    train[column + '_clean'] = train[column].apply(clean_text)

train[['prompt', 'prompt_clean']].head()

,prompt,prompt_clean
0,Pick the best possible answer: What is Martin ...,pick the best possible answer what is martin h...
1,What is accelerator-based light-ion fusion?,what is accelerator based light ion fusion
2,Determine the correct option: What is the term...,determine the correct option what is the term ...
3,Select the most accurate option: What is Marti...,select the most accurate option what is martin...
4,Identify the correct statement: What is the co...,identify the correct statement what is the con...


In [34]:
train['prompt_tokens'] = train['prompt_clean'].str.split()

train[['prompt_clean', 'prompt_tokens']].head()

,prompt_clean,prompt_tokens
0,pick the best possible answer what is martin h...,"[pick, the, best, possible, answer, what, is, ..."
1,what is accelerator based light ion fusion,"[what, is, accelerator, based, light, ion, fus..."
2,determine the correct option what is the term ...,"[determine, the, correct, option, what, is, th..."
3,select the most accurate option what is martin...,"[select, the, most, accurate, option, what, is..."
4,identify the correct statement what is the con...,"[identify, the, correct, statement, what, is, ..."


In [35]:
def average_precision_at_3(actual, predicted):
    predicted = predicted[:3]

    if actual not in predicted:
        return 0

    rank = predicted.index(actual) + 1

    return 1 / rank

In [36]:
def map_at_3(actual_answers, predictions):
    scores = []

    for actual, predicted in zip(actual_answers, predictions):
        score = average_precision_at_3(actual, predicted)
        scores.append(score)

    return np.mean(scores)

In [37]:
clean_columns = [
    'prompt_clean',
    'A_clean',
    'B_clean',
    'C_clean',
    'D_clean',
    'E_clean'
]

all_text = []

for column in clean_columns:
    all_text.extend(train[column].tolist())

print('Number of text samples:', len(all_text))

Number of text samples: 12000


In [38]:
tfidf = TfidfVectorizer(stop_words='english')

tfidf.fit(all_text)

print('TF-IDF vocabulary size:', len(tfidf.vocabulary_))

TF-IDF vocabulary size: 2762


In [39]:
option_labels = ['A', 'B', 'C', 'D', 'E']
tfidf_predictions = []
tfidf_scores = []

for _, row in train.iterrows():
    prompt_vector = tfidf.transform([row['prompt_clean']])

    options = [
        row['A_clean'],
        row['B_clean'],
        row['C_clean'],
        row['D_clean'],
        row['E_clean']
    ]

    option_vectors = tfidf.transform(options)

    similarities = cosine_similarity(
        prompt_vector,
        option_vectors
    )[0]

    order = np.argsort(similarities)[::-1]
    prediction = [option_labels[i] for i in order]

    tfidf_predictions.append(prediction)
    tfidf_scores.append(similarities)

In [40]:
train['tfidf_prediction'] = tfidf_predictions

tfidf_map = map_at_3(
    train['answer'].tolist(),
    tfidf_predictions
)

print('TF-IDF MAP@3:', round(tfidf_map, 4))

train[
    ['id', 'answer', 'tfidf_prediction']
].head(10)

TF-IDF MAP@3: 0.2762


,id,answer,tfidf_prediction
0,1,B,"[C, D, B, A, E]"
1,2,A,"[C, A, B, D, E]"
2,3,C,"[E, D, C, B, A]"
3,4,B,"[C, D, B, A, E]"
4,5,A,"[E, D, B, C, A]"
5,6,C,"[E, D, C, B, A]"
6,7,E,"[A, C, D, E, B]"
7,8,A,"[D, C, B, E, A]"
8,9,A,"[E, D, C, B, A]"
9,10,A,"[A, C, E, B, D]"


In [41]:
sentences = []

for column in clean_columns:
    for text in train[column]:
        sentences.append(text.split())

print('Number of sentences:', len(sentences))

Number of sentences: 12000


In [42]:
word2vec = Word2Vec(
    sentences=sentences,
    vector_size=100,
    window=5,
    min_count=2,
    workers=4,
    seed=42
)

print('Word2Vec vocabulary size:', len(word2vec.wv))

Word2Vec vocabulary size: 3003


In [43]:
def get_sentence_vector(text):
    vectors = []

    for word in text.split():
        if word in word2vec.wv:
            vectors.append(word2vec.wv[word])

    if len(vectors) == 0:
        return np.zeros(100)

    return np.mean(vectors, axis=0)

In [44]:
word2vec_predictions = []
word2vec_scores = []

for _, row in train.iterrows():
    prompt_vector = get_sentence_vector(
        row['prompt_clean']
    ).reshape(1, -1)

    option_vectors = []

    for label in option_labels:
        option_vector = get_sentence_vector(
            row[label + '_clean']
        )
        option_vectors.append(option_vector)

    option_vectors = np.array(option_vectors)

    similarities = cosine_similarity(
        prompt_vector,
        option_vectors
    )[0]

    order = np.argsort(similarities)[::-1]
    prediction = [option_labels[i] for i in order]

    word2vec_predictions.append(prediction)
    word2vec_scores.append(similarities)

In [45]:
train['word2vec_prediction'] = word2vec_predictions

word2vec_map = map_at_3(
    train['answer'].tolist(),
    word2vec_predictions
)

print('Word2Vec MAP@3:', round(word2vec_map, 4))

Word2Vec MAP@3: 0.3327


## milestone 3

knowledge dataset loaded

In [15]:
knowledge_data = load_dataset("allenai/openbookqa", "additional", split="train")

documents = []
for row in knowledge_data:
    fact = row.get("fact1", "")
    if fact and fact not in documents:
        documents.append(fact)

documents = documents[:2000]

print("Number of knowledge documents:", len(documents))
print(documents[0])

additional/train-00000-of-00001.parquet:   0%|          | 0.00/635k [00:00<?, ?B/s]

additional/validation-00000-of-00001.par(…):   0%|          | 0.00/75.9k [00:00<?, ?B/s]

additional/test-00000-of-00001.parquet:   0%|          | 0.00/72.5k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/4957 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/500 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/500 [00:00<?, ? examples/s]

Number of knowledge documents: 1294
the sun is the source of energy for physical cycles on Earth


In [16]:
embedding_model_name = "BAAI/bge-small-en-v1.5"
embedding_model = SentenceTransformer(embedding_model_name)

document_embeddings = embedding_model.encode(
    documents,
    show_progress_bar=True,
    normalize_embeddings=True
)

print(document_embeddings.shape)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/743 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/41 [00:00<?, ?it/s]

(1294, 384)


In [17]:
embedding_dim = document_embeddings.shape[1]
faiss_index = faiss.IndexFlatIP(embedding_dim)
faiss_index.add(document_embeddings.astype("float32"))

print("FAISS index size:", faiss_index.ntotal)


FAISS index size: 1294


In [23]:
#retrieval:
top_k = 3

prompt_embeddings = embedding_model.encode(
    train["prompt"].tolist(),
    show_progress_bar=True,
    normalize_embeddings=True
).astype("float32")

scores, indices = faiss_index.search(prompt_embeddings, top_k)

retrieved_documents = []
for row_indices in indices:
    docs = [documents[i] for i in row_indices]
    retrieved_documents.append(docs)

train["retrieved_documents"] = retrieved_documents
train["retrieved_context"] = ["\n".join(docs) for docs in retrieved_documents]

print("Question:")
print(train.loc[0, "prompt"])
print("\nRetrieved context:")
print(train.loc[0, "retrieved_context"])


Batches:   0%|          | 0/63 [00:00<?, ?it/s]

Question:
Pick the best possible answer: What is Martin Heidegger's view on the relationship between time and human existence? among the listed options.

Retrieved context:
feeling is when an living thing senses through touch
seconds are used to measure time
if a living thing dies then that living thing is dead


In [24]:
def build_augmented_prompt(row):
    return f"""Context:
{row['retrieved_context']}

Question:
{row['prompt']}

Options:
A {row['A']}
B {row['B']}
C {row['C']}
D {row['D']}
E {row['E']}

Return only the best answer."""

In [25]:
llm_name = "google/flan-t5-base"

tokenizer = AutoTokenizer.from_pretrained(llm_name)
llm = AutoModelForSeq2SeqLM.from_pretrained(llm_name)

device = "cuda" if torch.cuda.is_available() else "cpu"
llm = llm.to(device)

print("Using device:", device)

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Using device: cuda


In [46]:
sample_size = 25
sample = train.head(sample_size).copy()

generated_answers = []
rag_predictions = []

def get_answer_order(generated_text, row):
    text = generated_text.strip().upper()

    for label in option_labels:
        if text == label or text.startswith(label + " "):
            best = label
            break
    else:
        best = None
        for label in option_labels:
            option_text = str(row[label]).strip().upper()
            if option_text and (option_text in text or text in option_text):
                best = label
                break

    if best is None:
        best = "A"

    return [best] + [label for label in option_labels if label != best]

for _, row in sample.iterrows():
    prompt = build_augmented_prompt(row)
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=512
    ).to(device)

    outputs = llm.generate(
        **inputs,
        max_new_tokens=12
    )

    answer_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    prediction = get_answer_order(answer_text, row)

    generated_answers.append(answer_text)
    rag_predictions.append(prediction)

print("Finished inference for", sample_size, "questions")

Finished inference for 25 questions


In [47]:
actual_answers = sample["answer"].tolist()

rag_map3 = map_at_3(actual_answers, rag_predictions)
rag_accuracy = np.mean([
    actual == predicted[0]
    for actual, predicted in zip(actual_answers, rag_predictions)
])

print("MAP@3:", rag_map3)
print("Accuracy:", rag_accuracy)

for i in range(min(3, len(sample))):
    print("\nExample", i + 1)
    print("Retrieved context:")
    print(sample.iloc[i]["retrieved_context"])
    print("Generated answer:", generated_answers[i])
    print("Predicted answer:", rag_predictions[i][0])
    print("Actual answer:", actual_answers[i])

MAP@3: 0.4333333333333333
Accuracy: 0.28

Example 1
Retrieved context:
feeling is when an living thing senses through touch
seconds are used to measure time
if a living thing dies then that living thing is dead
Generated answer: B
Predicted answer: B
Actual answer: B

Example 2
Retrieved context:
a flashlight converts chemical energy into light energy
a star is a source of light energy through nuclear reactions
plasma is formed by electrons separating from atoms in stars
Generated answer: A
Predicted answer: A
Actual answer: A

Example 3
Retrieved context:
a star is a source of light energy through nuclear reactions
a star is a source of light through nuclear reactions
nuclear reactions in stars causes stars to produce light
Generated answer: B
Predicted answer: B
Actual answer: C


In [53]:
import os

input_files = []
for dirname, _, filenames in os.walk("/kaggle/input"):
    for filename in filenames:
        input_files.append(os.path.join(dirname, filename))

test_path = [p for p in input_files if p.endswith("test.csv")][0]
sample_path = [p for p in input_files if p.endswith("sample_submission.csv")][0]

test = pd.read_csv(test_path)
submission = pd.read_csv(sample_path)

test_embeddings = embedding_model.encode(
    test["prompt"].tolist(),
    show_progress_bar=True,
    normalize_embeddings=True
).astype("float32")

_, indices = faiss_index.search(test_embeddings, top_k)

test["retrieved_documents"] = [[documents[i] for i in row] for row in indices]
test["retrieved_context"] = ["\n".join(docs) for docs in test["retrieved_documents"]]

test_predictions = []

for _, row in test.iterrows():
    prompt = build_augmented_prompt(row)
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512).to(device)
    outputs = llm.generate(**inputs, max_new_tokens=12)
    answer_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    prediction = get_answer_order(answer_text, row)
    test_predictions.append(" ".join(prediction[:3]))

submission.iloc[:, -1] = test_predictions
submission.to_csv("/kaggle/working/submission.csv", index=False)

print(submission.head())
print("Saved /kaggle/working/submission.csv")

Batches:   0%|          | 0/16 [00:00<?, ?it/s]

   ID Prediction
0   1      D A B
1   2      A B C
2   3      A B C
3   4      B A C
4   5      D A B
Saved /kaggle/working/submission.csv


In [ ]:
#milestonme 3 google forms

In [9]:
!pip install faiss-cpu #install FAISS

import pandas as pd 
import numpy as np 
import faiss 
from sentence_transformers import SentenceTransformer, CrossEncoder 
from transformers import AutoTokenizer, pipeline 
from sklearn.feature_extraction.text import TfidfVectorizer 
from sklearn.metrics.pairwise import cosine_similarity 

train = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv') 

print("Creating nowledge base")
kb = [] 
for idx, row in train.iterrows(): 
    correct_letter = row['answer'] 
    kb.append(str(row[correct_letter])) 

print("Loading embedding model and creating index") 
model = SentenceTransformer('all-MiniLM-L6-v2') 
kb_embeddings = model.encode(kb, show_progress_bar=False) 
index = faiss.IndexFlatL2(kb_embeddings.shape[1]) 
index.add(kb_embeddings)

print("Knowledge base successfully created")

Creating nowledge base
Loading embedding model and creating index


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Knowledge base successfully created


In [10]:
zs = pipeline("zero-shot-classification", model="facebook/bart-large-mnli") 
row_150 = train.iloc[150] 
prompt_150 = str(row_150['prompt']) 
labels_150 = [str(row_150['A']), str(row_150['B']), str(row_150['C']), str(row_150['D']), str(row_150['E'])] 
ans_150 = str(row_150[row_150['answer']])


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [12]:
from transformers import pipeline
import pandas as pd


zs = pipeline(
    "zero-shot-classification",
    model="facebook/bart-large-mnli"
)

row = train.iloc[150]

prompt = str(row["prompt"])

labels = [
    str(row["A"]),
    str(row["B"]),
    str(row["C"]),
    str(row["D"]),
    str(row["E"])
]

correct_option = row["answer"]
correct_text = str(row[correct_option])

result = zs(prompt, candidate_labels=labels)

score = 0

for label, s in zip(result["labels"], result["scores"]):
    if label == correct_text:
        score = s
        break

print("Ground truth option:", correct_option)
print("Ground truth text:", correct_text)
print("Probability:", round(score, 3))

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

Ground truth option: C
Ground truth text: The butterfly effect is the phenomenon that a small change in the initial conditions of a dynamical framework can cause significant distinctions in subsequent states, as defined by Lorenz in his book "The Essence of Chaos."
Probability: 0.384


In [13]:
query = str(train.iloc[150]["prompt"])

query_embedding = model.encode([query], show_progress_bar=False)

k = 10

distances, indices = index.search(query_embedding, k)

print("Retrieved indices:", indices[0])

true_index = 150

if true_index in indices[0]:
    rank = list(indices[0]).index(true_index) + 1
    print("True document rank:", rank)
else:
    print("True document not found in top 10")

Retrieved indices: [ 663 1701 1269 1532  576  847 1693 1906  168  150]
True document rank: 10


In [14]:
from sentence_transformers import CrossEncoder

cross_encoder = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

docs_10 = [kb[i] for i in indices[0]]

pairs = [[prompt, doc] for doc in docs_10]

ce_scores = cross_encoder.predict(pairs)

reranked = sorted(
    zip(indices[0], ce_scores),
    key=lambda x: x[1],
    reverse=True
)

print("Reranked results:")
for rank, (idx, score) in enumerate(reranked, start=1):
    print(rank, idx, round(float(score), 4))

true_index = 150

for rank, (idx, score) in enumerate(reranked, start=1):
    if idx == true_index:
        print("\nTrue document rank:", rank)
        break

config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

Reranked results:
1 150 4.7585
2 847 4.7526
3 1693 4.7526
4 1906 4.7526
5 1269 4.7375
6 1532 4.7375
7 168 4.7072
8 576 4.687
9 663 4.6602
10 1701 4.6602

True document rank: 1


In [15]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

row = train.iloc[42]
prompt = str(row["prompt"])

query_embedding = model.encode([prompt], show_progress_bar=False)

k = 5
distances, indices = index.search(query_embedding, k)

docs = [kb[i] for i in indices[0]]

context = " ".join(docs)

text = f"Context: {context} Question: {prompt}"

tokens = tokenizer(text, add_special_tokens=True, truncation=False)

print("Total tokens:", len(tokens["input_ids"]))

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Total tokens: 216


In [16]:
from transformers import pipeline

zs = pipeline(
    "zero-shot-classification",
    model="facebook/bart-large-mnli"
)

row = train.iloc[150]

prompt = str(row["prompt"])

true_option = row["answer"]
true_document = kb[150]

labels = [
    str(row["A"]),
    str(row["B"]),
    str(row["C"]),
    str(row["D"]),
    str(row["E"])
]

rag_prompt = f"Context: {true_document} Question: {prompt}"

result = zs(rag_prompt, candidate_labels=labels)

correct_text = str(row[true_option])

score = 0

for label, s in zip(result["labels"], result["scores"]):
    if label == correct_text:
        score = s
        break

print("Ground truth option:", true_option)
print("Ground truth text:", correct_text)
print("Probability:", round(score, 3))

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

Ground truth option: C
Ground truth text: The butterfly effect is the phenomenon that a small change in the initial conditions of a dynamical framework can cause significant distinctions in subsequent states, as defined by Lorenz in his book "The Essence of Chaos."
Probability: 0.989


In [17]:
from transformers import pipeline

zs = pipeline(
    "zero-shot-classification",
    model="facebook/bart-large-mnli"
)

row = train.iloc[150]

prompt = str(row["prompt"])

labels = [
    str(row["A"]),
    str(row["B"]),
    str(row["C"]),
    str(row["D"]),
    str(row["E"])
]

correct_option = row["answer"]
correct_text = str(row[correct_option])

wrong_context = kb[999]

adversarial_prompt = f"Context: {wrong_context} Question: {prompt}"

result = zs(adversarial_prompt, candidate_labels=labels)

score = 0

for label, s in zip(result["labels"], result["scores"]):
    if label == correct_text:
        score = s
        break

print("Ground truth option:", correct_option)
print("Ground truth text:", correct_text)
print("Probability:", round(score, 3))

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

Ground truth option: C
Ground truth text: The butterfly effect is the phenomenon that a small change in the initial conditions of a dynamical framework can cause significant distinctions in subsequent states, as defined by Lorenz in his book "The Essence of Chaos."
Probability: 0.529


In [18]:
hits = 0
k = 5

for i in range(100):
    prompt = str(train.iloc[i]["prompt"])

    query_embedding = model.encode([prompt], show_progress_bar=False)

    distances, indices = index.search(query_embedding, k)

    docs = [kb[idx] for idx in indices[0]]

    correct_option = train.iloc[i]["answer"]
    correct_text = str(train.iloc[i][correct_option])

    if correct_text in docs:
        hits += 1

hit_rate = hits / 100 * 100

print("Hits:", hits)
print("Hit Rate:", round(hit_rate, 1), "%")

Hits: 73
Hit Rate: 73.0 %


In [19]:
def mapk(actual, predicted, k=3):
    for i, label in enumerate(predicted[:k]):
        if label == actual:
            return 1 / (i + 1)
    return 0

scores = []

for i in range(20):

    row = train.iloc[i]
    prompt = str(row["prompt"])

    query_embedding = model.encode([prompt], show_progress_bar=False)
    _, indices = index.search(query_embedding, 5)

    docs = [kb[idx] for idx in indices[0]]

    pairs = [[prompt, doc] for doc in docs]
    ce_scores = cross_encoder.predict(pairs)

    best_doc = docs[np.argmax(ce_scores)]

    rag_prompt = f"Context: {best_doc} Question: {prompt}"

    labels = [
        str(row["A"]),
        str(row["B"]),
        str(row["C"]),
        str(row["D"]),
        str(row["E"])
    ]

    result = zs(rag_prompt, candidate_labels=labels)

    option_map = {
        str(row["A"]): "A",
        str(row["B"]): "B",
        str(row["C"]): "C",
        str(row["D"]): "D",
        str(row["E"]): "E"
    }

    top3 = [option_map[label] for label in result["labels"][:3]]

    scores.append(mapk(row["answer"], top3))

print("MAP@3:", round(np.mean(scores), 3))

You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


MAP@3: 0.975
